In [1]:
# ============================================
# 1. ИМПОРТЫ И ПРОВЕРКИ
# ============================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Проверка GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

# ============================================
# 2. ПУТИ К ДАННЫМ
# ============================================
# Путь к твоему датасету
dataset_path = "/home/jetsonbot/jetbot/notebooks/my_teleop/my_teleop_dataset"

# Здесь должны быть папки free и blocked
free_path = os.path.join(dataset_path, "free")
blocked_path = os.path.join(dataset_path, "blocked")

print(f"📁 Папка free: {free_path}")
print(f"📁 Папка blocked: {blocked_path}")
print(f"Снимков free: {len(os.listdir(free_path)) if os.path.exists(free_path) else 0}")
print(f"Снимков blocked: {len(os.listdir(blocked_path)) if os.path.exists(blocked_path) else 0}")

if not os.path.exists(free_path) or not os.path.exists(blocked_path):
    raise Exception("❌ Папки free и blocked не найдены! Проверь пути.")

# ============================================
# 3. ПОДГОТОВКА ДАННЫХ
# ============================================
# Классы: 0 - free (можно ехать), 1 - blocked (препятствие)
class_names = ['free', 'blocked']
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

# Трансформации для изображений
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Кастомный Dataset
class JetBotDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.images = []
        self.labels = []
        self.transform = transform
        
        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.exists(class_dir):
                print(f"⚠️ Папка {class_dir} не найдена, пропускаем")
                continue
                
            for img_name in os.listdir(class_dir):
                # Игнорируем скрытые файлы и папки (начинаются с точки)
                if img_name.startswith('.'):
                    continue
                    
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(class_dir, img_name)
                    self.images.append(img_path)
                    self.labels.append(class_to_idx[class_name])
        
        print(f"✅ Загружено {len(self.images)} изображений")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Загружаем все данные
full_dataset = JetBotDataset(dataset_path, transform=transform)

if len(full_dataset) == 0:
    raise Exception("❌ Нет данных для обучения!")

# Разделяем на train и validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

print(f"📊 Train: {len(train_dataset)} изображений")
print(f"📊 Validation: {len(val_dataset)} изображений")

# Создаем загрузчики данных
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ============================================
# 4. СОЗДАНИЕ МОДЕЛИ
# ============================================
# Используем предобученную ResNet18
model = models.resnet18(pretrained=True)

# Заменяем последний слой на бинарную классификацию (free/blocked)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # 2 класса

model = model.to(device)

print("✅ Модель ResNet18 создана")

# ============================================
# 5. НАСТРОЙКА ОБУЧЕНИЯ
# ============================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

num_epochs = 30  # для 7 фоток хватит и 30 эпох

print(f"🎯 Начинаем обучение на {num_epochs} эпох")

# ============================================
# 6. ЦИКЛ ОБУЧЕНИЯ
# ============================================
train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(num_epochs):
    # Обучение
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Валидация
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    scheduler.step()
    
    print(f"Эпоха [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# ============================================
# 7. СОХРАНЕНИЕ МОДЕЛИ
# ============================================
model_path = "/home/jetsonbot/jetbot/notebooks/my_teleop/best_model_resnet18.pth"
torch.save(model.state_dict(), model_path)
print(f"✅ Модель сохранена в {model_path}")

# ============================================
# 8. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# ============================================
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Эпохи')
plt.ylabel('Loss')
plt.legend()
plt.title('График потерь')

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Val Accuracy')
plt.xlabel('Эпохи')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.title('График точности')

plt.tight_layout()
plt.show()

# ============================================
# 9. ТЕСТ НА ОДНОМ ИЗОБРАЖЕНИИ
# ============================================
def predict_image(image_path):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
    
    return predicted.item(), probabilities[0].cpu().numpy()

# Найдем одно изображение для теста
test_images = []
if os.path.exists(free_path):
    test_images.extend([os.path.join(free_path, f) for f in os.listdir(free_path)[:1]])
if os.path.exists(blocked_path):
    test_images.extend([os.path.join(blocked_path, f) for f in os.listdir(blocked_path)[:1]])

print("\n🔍 Тестирование модели:")
for img_path in test_images:
    pred_class, probs = predict_image(img_path)
    actual_class = 'free' if 'free' in img_path else 'blocked'
    print(f"Файл: {os.path.basename(img_path)}")
    print(f"  Реальный класс: {actual_class}")
    print(f"  Предсказанный: {class_names[pred_class]}")
    print(f"  Вероятности: free={probs[0]:.3f}, blocked={probs[1]:.3f}")

Используется устройство: cuda
📁 Папка free: /home/jetsonbot/jetbot/notebooks/my_teleop/my_teleop_dataset/free
📁 Папка blocked: /home/jetsonbot/jetbot/notebooks/my_teleop/my_teleop_dataset/blocked
Снимков free: 100
Снимков blocked: 104
✅ Загружено 204 изображений
📊 Train: 163 изображений
📊 Validation: 41 изображений
✅ Модель ResNet18 создана
🎯 Начинаем обучение на 30 эпох


/home/jetsonbot/.local/lib/python3.6/site-packages/torch/nn/functional.py:718: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at  /media/nvidia/NVME/pytorch/pytorch-v1.9.0/c10/core/TensorImpl.h:1156.)
  return torch.max_pool2d(input, kernel_size, stride, padding, dilation, ceil_mode)


Эпоха [1/30] Train Loss: 0.5982, Train Acc: 84.66% Val Loss: 2.5316, Val Acc: 60.98%
Эпоха [2/30] Train Loss: 0.2145, Train Acc: 93.25% Val Loss: 0.3272, Val Acc: 90.24%
Эпоха [3/30] Train Loss: 0.0864, Train Acc: 98.16% Val Loss: 0.0057, Val Acc: 100.00%
Эпоха [4/30] Train Loss: 0.0585, Train Acc: 96.93% Val Loss: 0.1803, Val Acc: 90.24%
Эпоха [5/30] Train Loss: 0.0839, Train Acc: 96.93% Val Loss: 0.0029, Val Acc: 100.00%
Эпоха [6/30] Train Loss: 0.0525, Train Acc: 98.16% Val Loss: 0.0044, Val Acc: 100.00%
Эпоха [7/30] Train Loss: 0.0284, Train Acc: 98.77% Val Loss: 0.0021, Val Acc: 100.00%
Эпоха [8/30] Train Loss: 0.0402, Train Acc: 98.16% Val Loss: 0.0027, Val Acc: 100.00%
Эпоха [9/30] Train Loss: 0.0153, Train Acc: 99.39% Val Loss: 0.0027, Val Acc: 100.00%
Эпоха [10/30] Train Loss: 0.0097, Train Acc: 100.00% Val Loss: 0.0021, Val Acc: 100.00%
Эпоха [11/30] Train Loss: 0.0076, Train Acc: 100.00% Val Loss: 0.0016, Val Acc: 100.00%
Эпоха [12/30] Train Loss: 0.0033, Train Acc: 100.00% 


🔍 Тестирование модели:
Файл: free_077.jpg
  Реальный класс: free
  Предсказанный: free
  Вероятности: free=1.000, blocked=0.000
Файл: blcked_030.jpg
  Реальный класс: blocked
  Предсказанный: blocked
  Вероятности: free=0.000, blocked=1.000
